# Inspect Aggregated SWE Datasets

HRU-level inspection of the four SWE sources. Mirrors [inspect_consolidated_swe.ipynb](../consolidated/inspect_consolidated_swe.ipynb).

Sources (see `catalog/variables.yml` → `snow_water_equivalent`):

- **Daymet V4 R1** `swe` (kg m⁻² ≡ mm, daily) — `region=na` aggregated NCs at `<project>/data/aggregated/daymet/`.
- **SNODAS** `swe` (kg m⁻² ≡ mm, daily) — at `<project>/data/aggregated/snodas/`.
- **ERA5-Land** `sd` (m water-eq, daily, `cell_methods: "time: point"`) — at `<project>/data/aggregated/era5_land_sd/`. Note the storage key `era5_land_sd` is synthetic; the catalog source key is `era5_land` (the same alias case PR #142 added `catalog_source_key` for).
- **Margulis WUS-SR** `SWE` (m, daily, WUS-only) — at `<project>/data/aggregated/margulis_wus_sr/`. Carries `fabric_scope.fabrics: [or]`; the SWE target builder drops it on every non-OR fabric, so the cross-source view on the gfv2 fabric will normally show **3 sources** (Daymet + SNODAS + ERA5-Land) and the Margulis cell will be empty by design.

The SWE target combines these via per-HRU per-**day** **NaN-aware min/max in absolute inches** (`multi_source_minmax`, no 0-1 normalisation, mapped to PRMS `pkwater_equiv`). Cross-source magnitudes feed the target bound directly. Unit chains differ per source:

- Daymet, SNODAS: mm → inches via `÷ 25.4`.
- ERA5-Land, Margulis: m → mm → inches via `× 1000 / 25.4`.

All four conversions happen in the notebook's per-source `to_inches` helpers, mirroring what `targets/swe.py` does for the final calibration target.

Companion to:

- [inspect_consolidated_swe.ipynb](../consolidated/inspect_consolidated_swe.ipynb) — pre-aggregation gridded NCs / zarr.
- [inspect_target_swe.ipynb](../targets/inspect_target_swe.ipynb) — post-combination per-HRU min/max bounds.

## Per-target conventions in this notebook

- All four aggregated NCs carry SWE in their **native** units (kg m⁻² for Daymet/SNODAS, m for ERA5-Land/Margulis). The aggregator's transformation policy (`docs/architecture/transformation-pipeline.md`) reserves linear unit conversions for `targets/`; this notebook applies the same chain inline for cross-source visual comparison.
- **Cadence is daily.** Unlike AET / runoff / recharge / soil moisture (monthly), SWE comparisons happen at single-day snapshots. `TARGET_DATE` defaults to **2010-03-01** (near peak CONUS pack); switch to a summer date to see the no-snow regime instead.
- **`cell_methods: "time: point"`** for every source: SWE is instantaneous, never accumulated. Monthly aggregates would be a mean over time, not a sum.
- **`stat_method` choice** differs per source per CLAUDE.md's policy: Daymet/SNODAS/ERA5-Land use the default `mean` (no pre-aggregate masking); Margulis also uses `mean` (no per-pixel flag masking).
- **Reference source for the shared colour scale**: SNODAS (highest spatial fidelity over CONUS within its coverage).
- **Margulis handling**: opens gracefully when absent; cells iterate over the loaded set so a missing source (in-flight aggregation, or non-OR fabric) drops out without breaking the notebook.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from _helpers import (
    area_weighted_mean,
    discover_aggregated,
    load_fabric,
    load_project_paths,
    lookup_hrus_by_points,
    nan_hru_count,
    open_year,
    open_year_range,
    plot_hru_choropleth,
    plot_nan_hrus,
    save_figure,
    unit_from_catalog,
)

# Edit me to point at a real project directory:
PROJECT_DIR = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets"
)

# Set True (and re-run) to populate docs/figures/aggregated/<project>/*.png
import _helpers
_helpers.SAVE_FIGURES = True
_helpers.PROJECT = PROJECT_DIR.name

project_dir, datastore_dir, fabric_cfg = load_project_paths(PROJECT_DIR)
fabric = load_fabric(fabric_cfg)

TARGET = "swe"
TARGET_DATE = "2010-03-01"  # near-peak CONUS SWE
TARGET_YEAR = 2010
TIME_SERIES_YEARS = range(2005, 2011)
MM_PER_M = 1000.0
MM_PER_INCH = 25.4

REPRESENTATIVE_POINTS = {
    "Sierra Nevada (deep maritime snowpack)": (-119.0, 38.0),
    "Colorado Rockies (alpine continental)": (-106.8, 39.5),
    "Cascades (PNW maritime)": (-121.5, 47.0),
    "Adirondacks (eastern shallow)": (-74.0, 44.0),
}

# Per-source contract. `catalog_key` is the catalog source name (which
# differs from the on-disk storage key only for era5_land_sd, per the
# PR #142 catalog_source_key pattern). `region` is consumed by
# discover_aggregated/open_year — Daymet is region-partitioned per
# aggregate/daymet.py, the others are not.
SOURCES = {
    "daymet": {
        "label": "Daymet V4 R1 (swe, NA)",
        "var": "swe",
        "catalog_key": "daymet",
        "region": "na",
        "to_inches": lambda da: da / MM_PER_INCH,
    },
    "snodas": {
        "label": "SNODAS (swe)",
        "var": "swe",
        "catalog_key": "snodas",
        "region": None,
        "to_inches": lambda da: da / MM_PER_INCH,
    },
    "era5_land_sd": {
        "label": "ERA5-Land (sd)",
        "var": "sd",
        "catalog_key": "era5_land",  # storage key != catalog key
        "region": None,
        "to_inches": lambda da: da * MM_PER_M / MM_PER_INCH,
    },
    "margulis_wus_sr": {
        "label": "Margulis WUS-SR (SWE, OR fabric only)",
        "var": "SWE",
        "catalog_key": "margulis_wus_sr",
        "region": None,
        "to_inches": lambda da: da * MM_PER_M / MM_PER_INCH,
    },
}

print(f"Project: {project_dir}")
print(f"Datastore: {datastore_dir}")
print(f"Fabric: {fabric_cfg['path']} ({len(fabric)} HRUs)")

## Load aggregated datasets

Each source is opened from `<project>/data/aggregated/<source_key>/<source_key>_[<region>_]<TARGET_YEAR>_agg.nc`. Sources whose aggregation has not yet been produced are skipped with a clear message; downstream cells iterate over the loaded set so missing sources drop out naturally. Margulis on a non-OR fabric (or while its aggregation is in flight) will land here as SKIP — both states are normal.

In [ ]:
opened = {}
for source_key, info in SOURCES.items():
    paths = discover_aggregated(project_dir, source_key, region=info.get("region"))
    if paths is None:
        print(
            f"SKIP {info['label']}: no aggregated NCs at "
            f"{project_dir}/data/aggregated/{source_key}/"
        )
        continue
    ds = open_year(project_dir, source_key, TARGET_YEAR, region=info.get("region"))
    info["units"] = unit_from_catalog(info["catalog_key"], info["var"])
    opened[source_key] = (ds, info)
    values = ds[info["var"]].isel(time=0).to_pandas()
    print(
        f"{info['label']}: vars={list(ds.data_vars)} | "
        f"time={ds.time.values[0]}..{ds.time.values[-1]} | "
        f"HRUs={ds.sizes.get(fabric_cfg['id_col'], 'unknown')} | "
        f"NaN HRUs (t=0): {nan_hru_count(values)} | "
        f"catalog units: {info['units']}"
    )

In [ ]:
for source_key, (ds, info) in opened.items():
    print(f"{'=' * 60}\n{info['label']}\n{'=' * 60}")
    display(ds)

## What the dataset summaries tell us

Quick checks before the figures:

- **Time spans.** All four are daily; for `TARGET_YEAR=2010` each NC carries 365 timesteps (366 in leap years). A short year is usually a fetch / consolidation issue (missing days) — for SNODAS that's normal in 2004-2007 (the [snodas_archive_gaps](docs/references/...) note in memory documents real day gaps from NOHRSC's archive).
- **HRU counts.** All present sources should match the fabric count exactly. A mismatch points at the aggregation running against a different fabric or a stale weights cache.
- **NaN HRUs at t=0.** SNODAS / Daymet have small NaN footprints (CONUS-edge HRUs). ERA5-Land has very few because it's global. Margulis is **non-trivial**: ~90% of the gfv2 fabric is outside the Western US footprint, so on the OR fabric Margulis covers everything but on the gfv2 fabric most HRUs come out NaN at aggregation time. This is structural, not a bug.
- **Catalog units.** Each row prints the unit from `catalog/sources.yml` (recipes lesson 9), not the on-disk attr. The synthetic `era5_land_sd` storage key looks up against `catalog_key="era5_land"` per the SOURCES dict's `catalog_key` field.

The next cell plots each present source's `TARGET_DATE` slice in **native units** — useful for catching consolidation issues but not directly comparable across sources because the colourbars span two different unit systems (mm vs m).

## Native-units map at TARGET_DATE

Per-source choropleth in the source's native units (Daymet/SNODAS in mm via kg m⁻², ERA5-Land/Margulis in m). Not directly comparable across sources — magnitudes will differ by ~1000× between the mm and m panels. Use the **normalized comparison** below (inches, shared colour scale) for cross-source magnitude agreement.

What to look for in the native-units view:

- **Geographic pattern.** All four should show snow concentrated in the western mountains + a thinner band across the Northern Plains and Northeast.
- **Flat-zero panels** for sources that should be snowy in early March → smoking gun for a missed reducer (the instantaneous reducer applied as accumulated would silently zero things out, though PR #146's catalog-driven `_VARIABLE_KIND` derivation guards against that for ERA5-Land).
- **Tile boundaries / stripes** → fill / scale issue at consolidation.

In [ ]:
def _select_day(ds, var, date):
    """Select a single day, robust to start-of-day vs noon vs end-of-day
    timestamping conventions across the four sources.
    """
    return ds[var].sel(time=date, method="nearest")


def _grid_dims(n):
    """Pick a 1xN, 2xN, or 2x2 layout depending on source count."""
    if n <= 2:
        return 1, n
    return 2, 2


if not opened:
    print("No sources available; skipping native-unit map.")
else:
    nrow, ncol = _grid_dims(len(opened))
    fig, axes = plt.subplots(nrow, ncol, figsize=(8 * ncol, 5.5 * nrow), squeeze=False)
    flat = axes.flatten()
    for idx, (source_key, (ds, info)) in enumerate(opened.items()):
        da = _select_day(ds, info["var"], TARGET_DATE)
        values = da.to_pandas()
        plot_hru_choropleth(
            flat[idx],
            fabric,
            values,
            cmap="Blues",
            title=f"{info['label']}\n{TARGET_DATE} | {info['units']}",
            units=info["units"],
        )
    for idx in range(len(opened), nrow * ncol):
        flat[idx].set_visible(False)
    fig.suptitle(f"SWE — native units, {TARGET_DATE}", fontsize=13, y=1.01)
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_native_units_map")
    plt.show()

## Inches comparison map at TARGET_DATE (shared colour scale)

Convert each source to **inches** and plot side-by-side on a shared colour scale derived from SNODAS (the reference source — highest spatial fidelity over CONUS within its coverage). All four are now in absolute inches at the same daily cadence — this is the figure where cross-source magnitude differences are physically meaningful, because they propagate straight into the per-HRU per-day min/max bound that `targets/swe.py` computes.

Expected ordering on a March-1 peak day:

- SNODAS typically **highest** in the deepest pack (assimilates airborne SWE observations).
- Daymet **comparable** to SNODAS but smoother (no assimilation; gauge-interpolated climatology).
- ERA5-Land **systematically lower** in the deepest mountains (0.1° / ~11 km cells under-resolve the deepest ridges) but comparable on lowland HRUs.
- Margulis (where present) **highest of all** in the WUS — it's the highest-fidelity product within its footprint (480 m, ensemble snow reanalysis with Landsat SCA assimilation).

In [ ]:
if opened:
    converted = {}
    for source_key, (ds, info) in opened.items():
        da = _select_day(ds, info["var"], TARGET_DATE)
        converted[source_key] = info["to_inches"](da).to_pandas()

    ref_key = "snodas" if "snodas" in converted else next(iter(converted))
    ref_vals = converted[ref_key].dropna().values
    if ref_vals.size == 0:
        all_vals = np.concatenate([s.dropna().values for s in converted.values()])
        vmin = float(np.percentile(all_vals, 2)) if all_vals.size else 0.0
        vmax = float(np.percentile(all_vals, 98)) if all_vals.size else 1.0
    else:
        vmin = float(np.percentile(ref_vals, 2))
        vmax = float(np.percentile(ref_vals, 98))

    nrow, ncol = _grid_dims(len(converted))
    fig, axes = plt.subplots(nrow, ncol, figsize=(8 * ncol, 5.5 * nrow), squeeze=False)
    flat = axes.flatten()
    for idx, (source_key, values) in enumerate(converted.items()):
        plot_hru_choropleth(
            flat[idx],
            fabric,
            values,
            vmin=vmin,
            vmax=vmax,
            cmap="Blues",
            title=f"{SOURCES[source_key]['label']}\n{TARGET_DATE} | inches",
            units="inches",
        )
    for idx in range(len(converted), nrow * ncol):
        flat[idx].set_visible(False)
    fig.suptitle(
        f"SWE — inches (PRMS pkwater_equiv), colour scale from {SOURCES[ref_key]['label']} — {TARGET_DATE}",
        fontsize=13, y=1.02,
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_normalized_comparison")
    plt.show()

## Cross-source HRU distribution (histogram)

Density curves let you see *distribution shape* at the chosen day independent of geography. Things to watch:

- **Tall zero spike.** Expected — most HRUs (southern CONUS / lowland) carry no snow on a typical March day.
- **Right-tail behaviour.** SNODAS and Margulis should have the heaviest right tail (deepest peaks captured at high resolution); ERA5-Land's tail should be thinnest because its 11 km cells smooth ridges.
- **Bimodality.** Mountain vs lowland HRUs form two natural modes — all four sources should reproduce this.
- **Unimodal Gaussian** would be wrong — SWE is fundamentally bimodal (snowy vs not) at the HRU scale in winter.

In [ ]:
if opened:
    fig, ax = plt.subplots(figsize=(10, 5))
    for source_key, values in converted.items():
        finite = values.dropna()
        if finite.empty:
            continue
        ax.hist(
            finite,
            bins=60,
            histtype="step",
            label=SOURCES[source_key]["label"],
            density=True,
            linewidth=2,
        )
    ax.set_xlabel("SWE (inches, PRMS pkwater_equiv)")
    ax.set_ylabel("Density")
    ax.set_title(f"Cross-source HRU SWE distribution — {TARGET_DATE}")
    ax.legend()
    save_figure(fig, f"{TARGET}_histogram")
    plt.show()

## Representative-HRU time series

Four snow-bearing regions × multi-year daily series. `lookup_hrus_by_points` resolves each (lon, lat) to the containing HRU. Plotted at native **daily** cadence (line thinly to keep multi-year traces legible).

Things to watch:

- **Phase agreement.** All sources should peak near March, melt out by July, and stay at zero through fall.
- **Amplitude ordering.** Margulis (where present) and SNODAS typically highest in the Sierra / Cascades; ERA5-Land systematically lower (resolution); Daymet between SNODAS and ERA5-Land.
- **Mid-winter rain-on-snow drawdowns.** Visible in the Cascades / Olympics as sudden mid-winter dips followed by recovery.
- **Year-to-year correlation.** Big snow years (2011 in PNW, 2017 in California) should be synchronised across all sources at the same HRU.
- **Adirondacks panel.** SNODAS / Daymet / ERA5-Land should track each other closely (no Margulis here — it's WUS-only by design).

In [ ]:
if opened:
    rep_hrus = lookup_hrus_by_points(fabric, REPRESENTATIVE_POINTS)
    print("Representative HRUs:", rep_hrus)

    series_data = {}
    for source_key, info in SOURCES.items():
        if source_key not in opened:
            continue
        ds_range = open_year_range(
            project_dir, source_key, TIME_SERIES_YEARS, region=info.get("region")
        )
        try:
            id_dim = fabric_cfg["id_col"]
            sel = ds_range[info["var"]].sel({id_dim: list(rep_hrus.values())}).load()
        finally:
            ds_range.close()
        series_data[source_key] = info["to_inches"](sel)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=False)
    id_dim = fabric_cfg["id_col"]
    for ax, (label, hru_id) in zip(axes.flat, rep_hrus.items()):
        for source_key, da in series_data.items():
            try:
                da_hru = da.sel({id_dim: hru_id})
            except KeyError:
                # Margulis on a non-OR fabric: HRU not in that source.
                continue
            if np.isnan(da_hru.values).all():
                continue
            ax.plot(
                da_hru.time,
                da_hru.values,
                label=SOURCES[source_key]["label"],
                linewidth=0.5,
            )
        ax.set_title(f"{label} (HRU {hru_id})")
        ax.set_ylabel("inches")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    fig.suptitle(
        f"SWE at representative HRUs — "
        f"{min(TIME_SERIES_YEARS)}-{max(TIME_SERIES_YEARS)} (daily, inches)"
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_time_series")
    plt.show()

## Coverage gaps

Each red HRU is a NaN value at `TARGET_DATE` for that source. Per CLAUDE.md's policy, the aggregator uses `stat_method="mean"` for all four SWE sources (no `pre_aggregate_hook`), so NaN propagates through any partial-coverage HRU and is *honest* about geometric gaps rather than imputed.

**NaN-aware combination at target stage.** Unlike AET (where the multi-source min/max requires *all* sources to be finite), the SWE target uses **NaN-aware** min/max (`np.fmin` / `np.fmax`): the bound is well-defined whenever *at least one* source is finite at the HRU/day. So the union of finite HRUs is the calibratable set, not the intersection — which is why **Margulis's WUS-only footprint is fine**: HRUs outside the WUS still get a bound from Daymet + SNODAS + ERA5-Land.

An HRU is only NaN in the combined SWE target when *every* source is NaN there. NN-fill (when applied) is the target-stage post-processing step on the combined bound — never on the aggregated NC.

In [ ]:
if opened:
    nrow, ncol = _grid_dims(len(opened))
    fig, axes = plt.subplots(nrow, ncol, figsize=(8 * ncol, 5.5 * nrow), squeeze=False)
    flat = axes.flatten()
    for idx, (source_key, (ds, info)) in enumerate(opened.items()):
        da = _select_day(ds, info["var"], TARGET_DATE)
        values = da.to_pandas()
        n_nan = nan_hru_count(values)
        print(
            f"{info['label']}: {n_nan} NaN HRUs "
            f"({100 * n_nan / len(fabric):.2f}%)"
        )
        plot_nan_hrus(
            flat[idx],
            fabric,
            values,
            title=(
                f"{info['label']}\nNaN HRUs (red) — {n_nan} of {len(fabric)}"
            ),
        )
    for idx in range(len(opened), nrow * ncol):
        flat[idx].set_visible(False)
    fig.suptitle(
        "Coverage gaps — combined via NaN-aware min/max in targets/swe.py; "
        "NN-fill is a target-stage step, not an aggregator step",
        fontsize=12, y=1.02,
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_coverage")
    plt.show()

## Magnitude sanity check at TARGET_DATE

HRU-area-weighted CONUS-mean SWE for each source at the same day. Per `feedback_validate_magnitudes`: |% diff| above ~30% between sources at the same day is plausible inter-source disagreement; an *order-of-magnitude* miss against the others is a smoking gun for a unit-conversion bug (mm vs m vs inches, or instantaneous-vs-accumulated reducer flip).

Margulis on the gfv2 fabric will report a much lower mean than the others because ~90% of the fabric is outside its WUS footprint (NaN, area-weighted out).

In [ ]:
if opened:
    print(f"{'Source':<45} {'mean (in)':>12} {'95th pct (in)':>14}")
    print("-" * 75)
    for source_key, values in converted.items():
        info = SOURCES[source_key]
        agg_mean = area_weighted_mean(values, fabric)
        finite = values.dropna().values
        p95 = float(np.percentile(finite, 95)) if finite.size else float("nan")
        print(f"{info['label']:<45} {agg_mean:>12.3f} {p95:>14.3f}")

## Why the four SWE sources differ

After area-weighted aggregation to HRUs and unit harmonisation to inches, the four sources still disagree visibly. None of this is a pipeline bug — each reflects a real difference in how the source was produced. The disagreement *is* the calibration bound.

- **Daymet V4 R1** — daily-gridded climatology from spatial interpolation of NWS COOP / SNOTEL precipitation gauges + a temperature-based snow-fraction model. SWE is a *derived diagnostic* (precip - melt accounting), not a direct measurement. 1 km LCC over NA.
- **SNODAS** — operational analysis: snow model forced by AWIPS hourly precipitation, assimilating airborne / satellite / ground SWE observations. Closest thing to ground truth over CONUS but only available 2003-present. 1 km WGS84 over CONUS.
- **ERA5-Land** — global reanalysis (ECMWF IFS land model forced by ERA5 atmosphere). No SWE assimilation; pure model. 0.1° (~11 km) global. Systematically under-resolves the deepest mountain ridges because of its coarse cells.
- **Margulis WUS-SR** — 32-year (water years 1985-2017) snow reanalysis specifically for the WUS, assimilating Landsat snow-covered area into an ensemble snow model. Highest spatial fidelity (~480 m) of the four sources within its footprint; no operational stream past 2017. Bound to OR fabric via `fabric_scope.fabrics: [or]` in the catalog.

**Calibration target implication.** The SWE target uses all four sources as a per-HRU per-day **NaN-aware** min/max (`np.fmin` / `np.fmax`) in absolute inches. The bound is well-defined whenever ≥1 source has data at the (HRU, day), so Margulis's WUS-only footprint doesn't shrink the calibratable set — it just contributes additional spread inside the WUS. On the gfv2 fabric (Margulis filtered out by `fabric_scope`), the bound is a 3-source min/max; on the OR fabric, it's a 4-source min/max. NN-fill (when applied) lands on the combined bound at target stage, not on the aggregated NC.

Inspect the post-combination view in [inspect_target_swe.ipynb](../targets/inspect_target_swe.ipynb).

In [ ]:
for source_key, (ds, _) in opened.items():
    ds.close()
opened.clear()